In [22]:
import datetime
import os
import pprint
from typing import Literal

import numpy as np
import torch
from tianshou.algorithm import PPO
from tianshou.algorithm.algorithm_base import Algorithm
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.optim import AdamOptimizerFactory, LRSchedulerFactoryLinear
from tianshou.data import Collector, CollectStats, ReplayBuffer, VectorReplayBuffer
from tianshou.highlevel.logger import LoggerFactoryDefault
from tianshou.trainer import OnPolicyTrainerParams
from tianshou.utils.net.common import ActorCritic, Net
from tianshou.utils.net.continuous import ContinuousActorProbabilistic, ContinuousCritic
from torch import nn
from torch.distributions import Distribution, Independent, Normal

from src.misc.mujoco_env import make_mujoco_env
from src.utils import export_onnx

In [2]:
task: str = "Ant-v5"
persistence_base_dir: str = "./logs"
seed: int = 0
buffer_size: int = 4096
lr: float = 3e-4
gamma: float = 0.99
epoch: int = 100
epoch_num_steps: int = 30000
collection_step_num_env_steps: int = 2048
update_step_num_repetitions: int = 10
batch_size: int = 64
num_training_envs: int = 8
num_test_envs: int = 10
return_scaling: bool = True
vf_coef: float = 0.25
ent_coef: float = 0.0
gae_lambda: float = 0.95
bound_action_method: Literal["clip", "tanh"] | None = "clip"
max_grad_norm: float = 0.5
eps_clip: float = 0.2
dual_clip: float | None = None
value_clip: bool = True
advantage_normalization: bool = False
recompute_adv: bool = True
render: float = 0.0
resume_path: str | None = None
resume_id: str | None = None
logger_type: str = "tensorboard"
watch: bool = False

In [3]:
hidden_sizes = [64, 64]
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
env, training_envs, test_envs = make_mujoco_env(
    task,
    seed,
    num_training_envs,
    num_test_envs,
    obs_norm=True,
)

state_shape = env.observation_space.shape or env.observation_space.n
action_shape = env.action_space.shape or env.action_space.n
max_action = env.action_space.high[0]

print(state_shape)
print(action_shape)
print(max_action)

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=69477) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


(105,)
(8,)
1.0


In [5]:
np.random.seed(seed)
torch.manual_seed(seed)

In [6]:
net_a = Net(
        state_shape=state_shape,
        hidden_sizes=hidden_sizes,
        activation=nn.Tanh,
    )
actor = ContinuousActorProbabilistic(
    preprocess_net=net_a,
    action_shape=action_shape,
    unbounded=True,
).to(device)
net_c = Net(
    state_shape=state_shape,
    hidden_sizes=hidden_sizes,
    activation=nn.Tanh,
)
critic = ContinuousCritic(preprocess_net=net_c).to(device)
actor_critic = ActorCritic(actor, critic)

In [7]:
for m in actor_critic.modules():
    if isinstance(m, torch.nn.Linear):
        # orthogonal initialization
        torch.nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        torch.nn.init.zeros_(m.bias)
        
# do last policy layer scaling, this will make initial actions have (close to)
# 0 mean and std, and will help boost performances,
# see https://arxiv.org/abs/2006.05990, Fig.24 for details
for m in actor.mu.modules():
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.zeros_(m.bias)
        m.weight.data.copy_(0.01 * m.weight.data)

In [8]:
optim = AdamOptimizerFactory(lr=lr)

optim.with_lr_scheduler_factory(
    LRSchedulerFactoryLinear(
        max_epochs=epoch,
        epoch_num_steps=epoch_num_steps,
        collection_step_num_env_steps=collection_step_num_env_steps,
    )
)

AdamOptimizerFactory[id=5009006208, lr_scheduler_factory=LRSchedulerFactoryLinear[num_epochs=100, epoch_num_steps=30000, collection_step_num_env_steps=2048], lr=0.0003, weight_decay=0, eps=1e-08, betas=(0.9, 0.999)]

In [9]:
def dist(loc_scale: tuple[torch.Tensor, torch.Tensor]) -> Distribution:
    loc, scale = loc_scale
    return Independent(Normal(loc, scale), 1)

In [10]:
policy = ProbabilisticActorPolicy(
    actor=actor,
    dist_fn=dist,
    action_scaling=True,
    action_bound_method=bound_action_method,
    action_space=env.action_space,
)

/Users/asd/Documents/dev/everything-i-reach-for/.venv/lib/python3.13/site-packages/tianshou/algorithm/modelfree/reinforce.py:152: UserWarning: action_scaling and action_bound_method are only intended to deal with unbounded model action space, but found actor model bound action space with max_action=1.0. Consider using unbounded=True option of the actor model, or set action_scaling to False and action_bound_method to None.
  warnings.warn(


In [11]:
algorithm: PPO = PPO(
    policy=policy,
    critic=critic,
    optim=optim,
    gamma=gamma,
    gae_lambda=gae_lambda,
    max_grad_norm=max_grad_norm,
    vf_coef=vf_coef,
    ent_coef=ent_coef,
    return_scaling=return_scaling,
    eps_clip=eps_clip,
    value_clip=value_clip,
    dual_clip=dual_clip,
    advantage_normalization=advantage_normalization,
    recompute_advantage=recompute_adv,
)

In [12]:
buffer: VectorReplayBuffer | ReplayBuffer
if num_training_envs > 1:
    buffer = VectorReplayBuffer(buffer_size, len(training_envs))
else:
    buffer = ReplayBuffer(buffer_size)

In [14]:
training_collector = Collector[CollectStats](
    algorithm, training_envs, buffer, exploration_noise=True
)
test_collector = Collector[CollectStats](algorithm, test_envs)

In [15]:
now = datetime.datetime.now().strftime("%y%m%d-%H%M%S")
algo_name = "ppo"
log_name = os.path.join(task, algo_name, str(seed), now)
log_path = os.path.join(persistence_base_dir, log_name)
logger_factory = LoggerFactoryDefault()
logger_factory.logger_type = "tensorboard"
logger = logger_factory.create_logger(
    log_dir=log_path,
    experiment_name=log_name,
    run_id=resume_id,
)

In [16]:
def save_best_fn(policy: Algorithm) -> None:
    state = {"model": policy.state_dict(), "obs_rms": training_envs.get_obs_rms()}
    torch.save(state, os.path.join(log_path, "policy.pth"))

In [17]:
%%capture
result = algorithm.run_training(
OnPolicyTrainerParams(
        training_collector=training_collector,
        test_collector=test_collector,
        max_epochs=epoch,
        epoch_num_steps=epoch_num_steps,
        update_step_num_repetitions=update_step_num_repetitions,
        test_step_num_episodes=num_test_envs,
        batch_size=batch_size,
        collection_step_num_env_steps=collection_step_num_env_steps,
        save_best_fn=save_best_fn,
        logger=logger,
        test_in_training=False,
    )
)

In [18]:
pprint.pprint(result)

InfoStats(update_step=1500,
          best_score=656.1296020192447,
          best_reward=656.1296020192447,
          best_reward_std=300.31371675918626,
          train_step=3072000,
          train_episode=np.int64(10402),
          test_step=351106,
          test_episode=np.int64(1010),
          timing=TimingStats(total_time=1440.9586019515991,
                             train_time=1440.9586019515991,
                             train_time_collect=0.0,
                             train_time_update=821.3548595905304,
                             test_time=0.0,
                             update_speed=2131.9141270535865))


In [25]:
test_envs.seed(seed)
test_collector.reset()
collector_stats = test_collector.collect(n_episode=num_test_envs, render=render)

In [24]:
%%capture
export_onnx(os.path.join(log_path, "policy.pth"), policy, state_shape)